In [1]:
from pathlib import Path

import numpy as np
from ls_mcmc import logging, sampling

from cardiac_electrophysiology import mcmc_builder, posterior_builder

In [2]:
posterior_settings = posterior_builder.PosteriorBuilderSettings(
    paths=posterior_builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/mean_angle_field.npy"),
        ground_truth_path=Path("../data/ground_truth_angle_field.npy"),
        log_file_path=Path("../results/lsbip_logfile.log"),
    ),
    prior_parameters=posterior_builder.PriorParameters(
        kappa=0.05,
        tau=5,
        seed=0,
    ),
    eikonal_parameters=posterior_builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=3,
        transversal_velocity=1,
    ),
    observation_parameters=posterior_builder.ObservationParameters(
        num_observations=100,
        noise_variance=1e-3,
        seed=0,
    ),
    logger_settings=posterior_builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)
builder = posterior_builder.PosteriorBuilder(posterior_settings)
posterior, additional_output = builder.build(return_additional_data=True)

In [3]:
map_estimate = np.load("../results/map_estimate.npy")
builder_settings = mcmc_builder.MCMCBuilderSettings(
    mcmc_model_settings=mcmc_builder.MCMCModelSettings(
        log_posterior=posterior,
        reference_point=map_estimate,
        step_width=1e-4,
        index_to_track=42,
    ),
    logging_settings=logging.LoggerSettings(
        do_printing=True,
        logfile_path=None,
    )
)
builder = mcmc_builder.MCMCBuilder(builder_settings)
mcmc_sampler = builder.build()

In [4]:
rng = np.random.default_rng(seed=0)
noise = rng.normal(0, 1e-2, size=map_estimate.shape)
offset = 1e-2
sampler_settings=sampling.SamplerRunSettings(
    num_samples=500,
    initial_state=additional_output.prior_mean_parameter,
    print_interval=1,
)
storage, outputs = mcmc_sampler.run(sampler_settings)

| Iteration   | Time        | Accept Rate    | Component 42   | Run_mean_C_42  | 
--------------------------------------------------------------------------------
| 0.000e+00   | 6.959e-04   | +1.000e+00     | +2.702e+00     | +2.702e+00     | 
| 1.000e+00   | 1.882e+00   | +1.000e+00     | +2.700e+00     | +2.701e+00     | 
| 2.000e+00   | 2.877e+00   | +1.000e+00     | +2.700e+00     | +2.701e+00     | 
| 3.000e+00   | 3.970e+00   | +7.500e-01     | +2.700e+00     | +2.701e+00     | 
| 4.000e+00   | 4.920e+00   | +8.000e-01     | +2.711e+00     | +2.703e+00     | 
| 5.000e+00   | 5.858e+00   | +6.667e-01     | +2.711e+00     | +2.704e+00     | 
| 6.000e+00   | 6.786e+00   | +5.714e-01     | +2.711e+00     | +2.705e+00     | 
| 7.000e+00   | 7.711e+00   | +5.000e-01     | +2.711e+00     | +2.706e+00     | 
| 8.000e+00   | 8.632e+00   | +4.444e-01     | +2.711e+00     | +2.706e+00     | 
| 9.000e+00   | 9.558e+00   | +5.000e-01     | +2.719e+00     | +2.708e+00     | 
| 1.000e+01   | 1